## Генерация PNG знаков разных размеров из SVG

### Что делает ноутбук
- Берёт SVG из `shared/signs/speed_png/`.
- Для каждого SVG создаёт папку класса `shared/signs/speed_png/pngs/<base>_<speed>/`.
- Рендерит SVG в PNG (через `rsvg-convert`) и сохраняет несколько вариантов разных размеров.

### Зачем это нужно
- Реальные кропы в приложении бывают очень разного размера.
- Эти PNG используются как “исходники” для генератора датасета классификатора скорости.

### Требования
- Должен быть установлен `rsvg-convert` (пакет `librsvg`).
  - macOS: `brew install librsvg`
  - Ubuntu: `sudo apt-get install librsvg2-bin`

### Вход
- `shared/signs/speed_png/*.svg` с именами вида `<base>_<speed>.svg`, например `3.24_50.svg`.

### Выход
- `shared/signs/speed_png/pngs/<base>_<speed>/<base>_<speed>_<size>.png`

### Важно
- `thumbnail()` в PIL изменяет изображение **in-place**, поэтому для каждого размера делаем `copy()`. 


In [ ]:
# Код ниже генерирует PNG разных размеров из SVG.
# Описание ноутбука — в markdown-ячейке выше.

import io
import os
import shutil
import subprocess
from pathlib import Path

from PIL import Image

# Проверяем, что доступен бинарник для рендера SVG -> PNG.
# На macOS обычно ставится так: `brew install librsvg`
# На Ubuntu: `sudo apt-get install librsvg2-bin`
rsvg = shutil.which("rsvg-convert")
if not rsvg:
    raise RuntimeError(
        "Не найден `rsvg-convert`. Установи librsvg (например: `brew install librsvg`)."
    )

# Размеры PNG (максимальная сторона). Можно добавлять/убирать.
# Важно: эти файлы потом пойдут в генерацию датасета.
sizes = [512, 256, 128, 64, 32]

# Где лежат исходные SVG.
svgs_path = "shared/signs/speed_png/"

# Собираем список SVG.
svgs = [p for p in os.listdir(svgs_path) if p.endswith(".svg")]

for svg_file in svgs:
    # 1) Рендерим SVG в PNG-байты.
    proc = subprocess.run(
        [rsvg, str(Path(svgs_path, svg_file))],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if proc.returncode != 0:
        err = proc.stderr.decode("utf-8", errors="ignore")
        raise RuntimeError(f"rsvg-convert упал на {svg_file}:\n{err}")

    # 2) Парсим имя файла: ожидаем `<base>_<speed>.svg`.
    name = svg_file.replace(".svg", "")
    parts = name.split("_")
    if len(parts) != 2:
        raise ValueError(
            f"Неожиданное имя SVG: {svg_file}. Ожидается формат `<base>_<speed>.svg`."
        )

    base = parts[0]
    speed = parts[1]

    # 3) Создаём папку класса: `pngs/<base>_<speed>`.
    class_dir = Path(svgs_path, "pngs", f"{base}_{speed}")
    class_dir.mkdir(parents=True, exist_ok=True)

    # 4) Открываем PNG-изображение, полученное из stdout.
    # Важно: Image.open ленивый, поэтому держим BytesIO.
    img0 = Image.open(io.BytesIO(proc.stdout))

    # 5) Сохраняем версии разных размеров.
    # ВАЖНО: thumbnail() изменяет объект "на месте", поэтому для каждого размера делаем copy().
    for size in sizes:
        img = img0.copy()
        img.thumbnail((size, size))
        img = img.convert("RGBA")

        out_path = class_dir / f"{base}_{speed}_{size}.png"
        img.save(out_path)

print(
    f"[OK] Готово. SVG файлов: {len(svgs)}. Классы сохранены в: {Path(svgs_path) / 'pngs'}"
)
